# 01 — MLflow Tracing & Autologging for GenAI Agents

This notebook introduces **MLflow Tracing** — the observability layer for GenAI agents.

You'll learn:
* What an MLflow **trace** is and why it matters
* How **autologging** automatically captures agent execution with zero code changes
* How to inspect traces programmatically (spans, inputs, outputs, latency)

We build a simple tool-calling agent using the **OpenAI SDK** (pointed at a Databricks
model serving endpoint) and observe what MLflow captures.

**Compute:** DBR 18.2 ML cluster (`mlflow-eval-suite`)  
**Dependencies:** All pre-installed — no `%pip install` required.

In [0]:
# No installs needed — DBR 18.2 ML ships with mlflow 3.x, openai, and databricks-agents.
import mlflow
print(f"MLflow {mlflow.__version__} — ready to go.")

MLflow 3.8.1 — ready to go.


## What is an MLflow Trace?

A **trace** is a complete record of what happened during a single agent invocation. It captures:
* **Spans** — individual steps (LLM calls, tool executions, chain operations)
* **Inputs/Outputs** — what went in and came out of each step
* **Timing** — how long each step took
* **Parent-child relationships** — which steps triggered which

Think of it like a detailed execution log that lets you debug, evaluate, and monitor your agent.

```
Trace
├── Span: Agent (CHAIN)
│   ├── Span: LLM Call (CHAT_MODEL)     ← model decides to call a tool
│   ├── Span: Tool Execution (TOOL)     ← the tool runs and returns data
│   └── Span: LLM Call (CHAT_MODEL)     ← model synthesizes the final answer
```

In [0]:
"""Enable MLflow autologging for OpenAI SDK calls."""
import mlflow
import openai
import json
from mlflow.entities import SpanType

# Autolog instruments OpenAI SDK calls — every chat completion becomes a span.
mlflow.openai.autolog()

# Set an experiment to organize our traces
_username = (
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
)
EXPERIMENT_NAME = f"/Users/{_username}/agentic-evals-intro"
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"\u2713 Autologging enabled (MLflow {mlflow.__version__})")
print(f"  Experiment: {EXPERIMENT_NAME}")

2026/09/17 14:38:11 INFO mlflow.tracking.fluent: Experiment with name '/Users/adam.m.lang@gmail.com/agentic-evals-intro' does not exist. Creating a new experiment.


✓ Autologging enabled (MLflow 3.8.1)
  Experiment: /Users/adam.m.lang@gmail.com/agentic-evals-intro


## Step 1: Define a Tool

A **tool** is a function the agent can call to take action or retrieve data.
We define it as an OpenAI-compatible tool schema (JSON) and a plain Python function.

When the agent calls this tool, we’ll record it as a `TOOL` span in the trace.

In [0]:
"""Define a tool: schema for the LLM + Python implementation."""

# OpenAI tool schema — tells the LLM what the tool does and its parameters
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "lookup_order",
            "description": "Look up the status of a customer order by order ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "The order ID (e.g. ORD-1001)",
                    }
                },
                "required": ["order_id"],
            },
        },
    }
]

## mock database as python dict
def lookup_order(order_id: str) -> str:
    """Execute the lookup_order tool (simulated order database)."""
    orders = {
        "ORD-1001": "Shipped \u2014 arrives Thursday",
        "ORD-1002": "Processing \u2014 payment confirmed, preparing for shipment",
        "ORD-1003": "Delivered \u2014 left at front door on Monday",
        "ORD-1004": "Cancelled \u2014 refund issued",
    }
    return orders.get(order_id, f"Order {order_id} not found in system")


# Quick test
print(f"Tool test: lookup_order('ORD-1001') = '{lookup_order('ORD-1001')}'")
print(f"Tool schema registered: {TOOLS[0]['function']['name']}")

Tool test: lookup_order('ORD-1001') = 'Shipped — arrives Thursday'
Tool schema registered: lookup_order


## Step 2: Build the Agent

We use the **OpenAI SDK** pointed at a Databricks model serving endpoint.
The agent is a simple function: call LLM → if tool calls, execute them → call LLM again.

No framework needed — just the OpenAI client and a loop.

### Important code below
1. This `@mlflow.trace(span_type=SpanType.AGENT)` is the specific trace decorator for the Agentic application.

2. We always set the span type with this block:
- Specific span types supported: https://mlflow.org/docs/latest/genai/concepts/span/

```
# Execute tool with MLflow TOOL span
            with mlflow.start_span(name=fn_name, span_type=SpanType.TOOL) as span:
                span.set_inputs(args)
                result = TOOL_FUNCTIONS[fn_name](**args)
                span.set_outputs({"result": result})

            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})
```

In [0]:
"""Build a simple tool-calling agent using the OpenAI SDK + Databricks endpoint."""

MODEL_ENDPOINT = "databricks-llama-4-maverick"

# OpenAI client pointed at Databricks model serving
client = openai.OpenAI(
    base_url=(
        f"https://{dbutils.notebook.entry_point.getDbutils().notebook().getContext().browserHostName().get()}"
        "/serving-endpoints"
    ),
    api_key=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get(),
)

# Tool dispatch map
TOOL_FUNCTIONS = {"lookup_order": lookup_order}


@mlflow.trace(span_type=SpanType.AGENT)
def run_agent(user_message: str) -> str:
    """Simple tool-calling agent: LLM → tool execution → LLM."""
    messages = [{"role": "user", "content": user_message}]

    response = client.chat.completions.create(
        model=MODEL_ENDPOINT, messages=messages, tools=TOOLS
    )
    assistant_msg = response.choices[0].message

    # If tool calls requested, execute and get final answer
    if assistant_msg.tool_calls:
        messages.append({
            "role": "assistant",
            "content": assistant_msg.content or "",
            "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in assistant_msg.tool_calls
            ],
        })

        for tool_call in assistant_msg.tool_calls:
            fn_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)

            # Execute tool with MLflow TOOL span
            with mlflow.start_span(name=fn_name, span_type=SpanType.TOOL) as span:
                span.set_inputs(args)
                result = TOOL_FUNCTIONS[fn_name](**args)
                span.set_outputs({"result": result})

            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

        # Final LLM call with tool results
        response = client.chat.completions.create(model=MODEL_ENDPOINT, messages=messages)
        return response.choices[0].message.content

    return assistant_msg.content


print("\u2713 Agent ready")
print(f"  LLM: {MODEL_ENDPOINT}")
print(f"  Tools: [lookup_order]")

✓ Agent ready
  LLM: databricks-llama-4-maverick
  Tools: [lookup_order]


## Step 3: Run the Agent (Trace is Captured Automatically)

When we invoke the agent, MLflow autologging captures the entire execution as a trace.
No extra code needed \u2014 just call the agent normally.

In [0]:
"""Invoke the agent — MLflow captures the trace automatically."""

result = run_agent("What's the status of order ORD-1001?")
print(f"Agent response: {result}")

Agent response: The status of order ORD-1001 is "Shipped — arrives Thursday".


Trace(trace_id=tr-52b7f27d9ecf956ecf4706541d6d797b)

## Step 4: Inspect the Trace

Now let's look at what MLflow captured. The trace contains **spans** showing each step the agent
took: receiving the message, calling the LLM, executing the tool, and synthesizing the answer.

In [0]:
"""Retrieve and inspect the MLflow trace from our agent invocation."""

# Use local file tracking — serverless compute can't reach remote MLflow storage
mlflow.set_tracking_uri("file:///tmp/mlflow_traces")
mlflow.set_experiment("agent-traces-local")

# Re-run agent to generate a trace under local tracking
run_agent("What's the status of order ORD-1001?")

# Get the most recent trace from our experiment
ml_client = mlflow.tracking.MlflowClient()
experiment = mlflow.get_experiment_by_name("agent-traces-local")
traces = ml_client.search_traces(
    experiment_ids=[experiment.experiment_id],
    max_results=1,
    order_by=["timestamp_ms DESC"],
)
trace = traces[0]

# === TRACE OVERVIEW ===
print("=" * 60)
print("TRACE OVERVIEW")
print("=" * 60)
print(f"  Trace ID:  {trace.info.trace_id}")
print(f"  Duration:  {trace.info.execution_duration}ms")
print(f"  Status:    {trace.info.status}")
print(f"  # Spans:   {len(trace.data.spans)}")

# === SPAN DETAILS ===
hr = "=" * 60
print(f"\n{hr}")
print("SPANS (execution steps)")
print(hr)
for i, span in enumerate(trace.data.spans):
    indent = "  " if span.parent_id else ""
    print(f"\n{indent}[{i + 1}] {span.name}")
    print(f"{indent}    Type:     {span.span_type}")
    print(f"{indent}    Status:   {span.status.status_code}")
    if span.inputs:
        inp_str = str(span.inputs)[:120]
        print(f"{indent}    Inputs:   {inp_str}...")
    if span.outputs:
        out_str = str(span.outputs)[:120]
        print(f"{indent}    Outputs:  {out_str}...")

print(f"\n{hr}")
print("KEY TAKEAWAY")
print(hr)
print("  The trace shows the full execution flow:")
print("  1. Agent receives user message")
print("  2. LLM decides to call lookup_order tool")
print("  3. Tool executes and returns result (TOOL span)")
print("  4. LLM synthesizes final answer")
print("\n  All captured automatically!")

TRACE OVERVIEW
  Trace ID:  tr-f18df5c6fe1736882ae01d6be8c7c36a
  Duration:  1198ms
  Status:    TraceStatus.OK
  # Spans:   4

SPANS (execution steps)

[1] run_agent
    Type:     AGENT
    Status:   SpanStatusCode.OK
    Inputs:   {'user_message': "What's the status of order ORD-1001?"}...
    Outputs:  The status of order ORD-1001 is "Shipped — arrives Thursday"....

  [2] Completions
      Type:     CHAT_MODEL
      Status:   SpanStatusCode.OK
      Inputs:   {'model': 'databricks-llama-4-maverick', 'messages': [{'role': 'user', 'content': "What's the status of order ORD-1001?"...
      Outputs:  {'id': 'chatcmpl_60c648a6-54e8-4451-ae05-6c5e63a06eaa_756687e3-a935-483a-8142-b191f78053fd', 'choices': [{'finish_reason...

  [3] lookup_order
      Type:     TOOL
      Status:   SpanStatusCode.OK
      Inputs:   {'order_id': 'ORD-1001'}...
      Outputs:  {'result': 'Shipped — arrives Thursday'}...

  [4] Completions
      Type:     CHAT_MODEL
      Status:   SpanStatusCode.OK
      Inp

/home/spark-fd6c023e-da4e-4b9b-bb41-92/.ipykernel/1925/command-6554367973409800-3466223053:13: FutureWarning: Parameter 'experiment_ids' is deprecated. Please use 'locations' instead.
  traces = ml_client.search_traces(


Trace(trace_id=tr-f18df5c6fe1736882ae01d6be8c7c36a)

## Step 5: Compare \u2014 A Question That Doesn't Need the Tool

Let's ask something the agent can answer directly (no tool call needed).
Notice how the trace structure is simpler \u2014 fewer spans, no TOOL span.

In [0]:
"""Ask a question that doesn't require the tool — observe the simpler trace."""

result_no_tool = run_agent("What is a shipping tracking number?")
print(f"Agent response: {result_no_tool[:200]}")

# Get the new trace
traces_new = ml_client.search_traces(
    experiment_ids=[experiment.experiment_id],
    max_results=1,
    order_by=["timestamp_ms DESC"],
)
trace_no_tool = traces_new[0]

print(f"\n--- Trace comparison ---")
print(f"  With tool call:    {len(trace.data.spans)} spans")
print(f"  Without tool call: {len(trace_no_tool.data.spans)} spans")
print(f"\n  When the agent skips the tool, the trace is shorter (no TOOL span).")
print(f"  This is exactly what evaluation judges inspect to assess tool-call quality.")

Agent response: A shipping tracking number, also known as a tracking number or tracking ID, is a unique identifier assigned to a shipment by a shipping carrier, such as USPS, UPS, or FedEx. It allows the shipper, the

--- Trace comparison ---
  With tool call:    4 spans
  Without tool call: 2 spans

  When the agent skips the tool, the trace is shorter (no TOOL span).
  This is exactly what evaluation judges inspect to assess tool-call quality.


/home/spark-fd6c023e-da4e-4b9b-bb41-92/.ipykernel/1925/command-6554367973409802-1605766774:7: FutureWarning: Parameter 'experiment_ids' is deprecated. Please use 'locations' instead.
  traces_new = ml_client.search_traces(


Trace(trace_id=tr-848dd57c1d62633dd76d53570dca3753)

## Summary

What you learned:
* `mlflow.openai.autolog()` — one line to capture all OpenAI SDK calls as traces
* `@mlflow.trace` + `mlflow.start_span` — manual spans for tool execution
* **Traces** contain **spans** — each span is one step (LLM call, tool execution, etc.)
* You can retrieve and inspect traces programmatically via `MlflowClient`

**Next:** In notebook 02, we use MLflow’s built-in scorers (`ToolCallEfficiency`, `Safety`) to
automatically evaluate whether the agent calls the right tool for each question.